## 0. Install Dependencies

In [1]:
!pip install -q faiss-cpu sentence-transformers groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.8 MB/s eta 0:00:00


In [3]:
import os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = '/content/drive/MyDrive/augle_ai'

from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

COCO_PATH        = f'{BASE}/data/annotations/instances.json'
FAISS_INDEX_PATH = f'{BASE}/data/rag/faiss.index'
CHUNKS_PATH      = f'{BASE}/data/rag/chunks.json'

print(f'COCO path: {COCO_PATH}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COCO path: /content/drive/MyDrive/augle_ai/data/annotations/instances.json


In [15]:
import json
import math
import os
from pathlib import Path

import faiss
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
GROQ_MODEL       = 'llama-3.3-70b-versatile'
TOP_K            = 5


##  COCO JSON TO Text Documents

In [16]:
def build_chunks(coco_path: str) -> list:

    with open(coco_path) as f:
        coco = json.load(f)

    cat_map      = {c['id']: c['name'] for c in coco['categories']}
    ann_by_image = {}
    for ann in coco['annotations']:
        ann_by_image.setdefault(ann['image_id'], []).append(ann)

    chunks = []
    for img in coco['images']:
        anns = ann_by_image.get(img['id'], [])
        if not anns:
            continue
        ann_desc = []
        for a in anns:
            cat = cat_map.get(a['category_id'], 'unknown')
            x, y, w, h = a['bbox']
            ann_desc.append(
                f"{cat} at ({x:.0f},{y:.0f}) size {w:.0f}x{h:.0f} "
                f"area={w*h:.0f}px2 score={a.get('score', 0):.2f}"
            )
        chunks.append({
            'text': (
                f"Image {img['file_name']} (id={img['id']}, "
                f"{img['width']}x{img['height']}px) contains "
                f"{len(anns)} annotation(s): " + '; '.join(ann_desc) + '.'
            ),
            'image_id':  img['id'],
            'file_name': img['file_name'],
        })

    cat_counts = {}
    for ann in coco['annotations']:
        name = cat_map[ann['category_id']]
        cat_counts[name] = cat_counts.get(name, 0) + 1
    cat_summary = ', '.join(
        f'{k}:{v}' for k, v in sorted(cat_counts.items(), key=lambda x: -x[1])
    )
    chunks.insert(0, {
        'text': (
            f"Dataset dets: {len(coco['images'])} images, "
            f"{len(coco['annotations'])} annotations, "
            f"{len(coco['categories'])} categories. "
            f"Category distribution: {cat_summary}."
        ),
        'image_id':  -1,
        'file_name': 'Dataset Details',
    })

    print(f'[RAG] Built {len(chunks)} chunks')
    return chunks


print('Chunking function')

Chunking function


##Embedding + FAISS Index

In [17]:
def build_faiss_index(chunks, embed_model):
    texts      = [c['text'] for c in chunks]
    embeddings = embed_model.encode(texts, show_progress_bar=True,
                                    normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype=np.float32)
    index      = faiss.IndexFlatIP(embeddings.shape[1])  # cosine sim for L2-normalised vecs
    index.add(embeddings)
    print(f'[RAG] FAISS index: {index.ntotal} vectors, dim={embeddings.shape[1]}')
    return index


def save_index(index, chunks):
    Path(FAISS_INDEX_PATH).parent.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, FAISS_INDEX_PATH)
    with open(CHUNKS_PATH, 'w') as f:
        json.dump(chunks, f, indent=2)
    print(f'[RAG] Saved → {FAISS_INDEX_PATH}')


def load_index():
    index = faiss.read_index(FAISS_INDEX_PATH)
    with open(CHUNKS_PATH) as f:
        chunks = json.load(f)
    print(f'[RAG] Loaded index: {index.ntotal} vectors')
    return index, chunks


def retrieve(query, index, chunks, embed_model, k=TOP_K):
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_vec, k)
    return [
        {'text': chunks[i]['text'], 'score': float(scores[0][j])}
        for j, i in enumerate(idxs[0]) if i < len(chunks)
    ]


print('FAISS functions')

FAISS functions


#TOOLS
**Tool 1** — compute_bbox_stats: per-category area stats, flags tiny boxes  
**Tool 2** — compute_class_distribution: annotation counts, balance score, entropy

In [18]:
def compute_bbox_stats(coco_path, category_filter='', size_threshold_px=500):
    # Tool 1: Bounding-box area statistics per category.

    with open(coco_path) as f:
        coco = json.load(f)

    cat_map      = {c['id']: c['name'] for c in coco['categories']}
    stats_by_cat = {}
    suspicious   = []

    for ann in coco['annotations']:
        cat_name = cat_map.get(ann['category_id'], 'unknown')
        if category_filter and cat_name.lower() != category_filter.lower():
            continue
        x, y, w, h = ann['bbox']
        area = w * h
        stats_by_cat.setdefault(cat_name, {'areas': [], 'count': 0})
        stats_by_cat[cat_name]['areas'].append(area)
        stats_by_cat[cat_name]['count'] += 1
        if area < size_threshold_px:
            suspicious.append({
                'ann_id':   ann['id'],
                'image_id': ann['image_id'],
                'category': cat_name,
                'area':     area,
                'bbox':     ann['bbox'],
            })

    summary = {}
    for cat, data in stats_by_cat.items():
        areas = data['areas']
        mean  = sum(areas) / len(areas)
        summary[cat] = {
            'count':     data['count'],
            'mean_area': round(mean, 1),
            'min_area':  round(min(areas), 1),
            'max_area':  round(max(areas), 1),
            'std_area':  round((sum((a - mean)**2 for a in areas) / len(areas))**0.5, 1),
        }

    return {
        'threshold_px':     size_threshold_px,
        'category_filter':  category_filter or 'all',
        'per_category':     summary,
        'suspicious_count': len(suspicious),
        'suspicious_items': suspicious[:20],
    }


def compute_class_distribution(coco_path):

    # Tool 2: Annotation counts per category.
    with open(coco_path) as f:
        coco = json.load(f)

    cat_map = {c['id']: c['name'] for c in coco['categories']}
    counts  = {c['name']: 0 for c in coco['categories']}
    for ann in coco['annotations']:
        counts[cat_map.get(ann['category_id'], 'unknown')] = \
            counts.get(cat_map.get(ann['category_id'], 'unknown'), 0) + 1

    total   = sum(counts.values()) or 1
    probs   = [v / total for v in counts.values()]
    entropy = -sum(p * math.log2(p) for p in probs if p > 0)
    max_ent = math.log2(max(len(counts), 1))

    sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
    count_vals    = [v for _, v in sorted_counts]
    balance_score = min(count_vals) / max(count_vals) if max(count_vals) > 0 else 1.0

    return {
        'total_annotations':  total,
        'num_categories':     len(counts),
        'per_category':       dict(sorted_counts),
        'balance_score':      round(balance_score, 4),
        'normalized_entropy': round(entropy / max_ent if max_ent > 0 else 0.0, 4),
        'most_common':        sorted_counts[0]  if sorted_counts else ('', 0),
        'least_common':       sorted_counts[-1] if sorted_counts else ('', 0),
        'interpretation': (
            'Well balanced'          if balance_score > 0.5 else
            'Moderate imbalance'     if balance_score > 0.2 else
            'Severe imbalance — consider re-collecting data'
        ),
    }


TOOL_DEF = [
    {
        'type': 'function',
        'function': {
            'name': 'compute_bbox_stats',
            'description': (
                'Parses the COCO annotation file and computes bounding-box area '
                'statistics per category. Flags annotations whose area is below '
                'a given pixel threshold as suspicious (likely annotation errors).'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'category_filter':   {'type': 'string',  'description': 'Optional category name to filter. Empty = all.'},
                    'size_threshold_px': {'type': 'integer', 'description': 'Area threshold in pixels. Default 500.'},
                },
                'required': [],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'compute_class_distribution',
            'description': (
                'Returns annotation counts per category, a balance score '
                '(0=very imbalanced, 1=perfectly balanced), and normalised entropy.'
            ),
            'parameters': {'type': 'object', 'properties': {}, 'required': []},
        },
    },
]


def dispatch_tool(tool_name, tool_args, coco_path):
    if tool_name == 'compute_bbox_stats':
        result = compute_bbox_stats(
            coco_path,
            category_filter=tool_args.get('category_filter', ''),
            size_threshold_px=tool_args.get('size_threshold_px', 500),
        )
    elif tool_name == 'compute_class_distribution':
        result = compute_class_distribution(coco_path)
    else:
        result = {'error': f'Unknown tool: {tool_name}'}
    return json.dumps(result, indent=2)


print('Tools defined.')

Tools defined.


## RAG  

In [20]:
class RAGDataAuditor:
    def __init__(self, coco_path: str, rebuild_index: bool = False):
        self.coco_path   = coco_path
        self.groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

        print('[RAG] Loading embedding model...')
        self.embed_model = SentenceTransformer(EMBED_MODEL_NAME)

        index_exists = Path(FAISS_INDEX_PATH).exists() and Path(CHUNKS_PATH).exists()
        if index_exists and not rebuild_index:
            self.index, self.chunks = load_index()
        else:
            print('[RAG] Building index from COCO file...')
            self.chunks = build_chunks(coco_path)
            self.index  = build_faiss_index(self.chunks, self.embed_model)
            save_index(self.index, self.chunks)

    def query(self, user_query: str, verbose: bool = True) -> str:
        if verbose:
            print(f"\nQuery: {user_query}\n")

        # Step 1: retrieve relevant chunks
        retrieved = retrieve(user_query, self.index, self.chunks, self.embed_model)
        context   = '\n\n'.join(
            f'[Chunk {i+1}]\n{r["text"]}' for i, r in enumerate(retrieved)
        )
        if verbose:
            print(f'\n[Retrieved {len(retrieved)} chunks]')
            for i, r in enumerate(retrieved):
                print(f'  {i+1}. (score={r["score"]:.3f}) {r["text"][:100]}...')

        # Step 2: initial LLM call
        system_prompt = (
            'You are an expert data quality auditor for computer vision datasets. '
            'You have retrieved COCO dataset context and two tools to compute precise statistics. '
            'Always ground your answers in the retrieved context or tool outputs. '
            'Be concise and technically precise.\n\n'
            f'RETRIEVED CONTEXT:\n{context}'
        )
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_query},
        ]

        response = self.groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            tools=TOOL_DEF,
            tool_choice='auto',
            max_tokens=2048,
            temperature=0.1,
        )

        # Step 3: agentic tool-call loop (max 5 rounds)
        for _ in range(5):
            msg = response.choices[0].message
            if not msg.tool_calls:
                break

            if verbose:
                for tc in msg.tool_calls:
                    print(f'\n[Tool Call] {tc.function.name}({tc.function.arguments})')

            messages.append({
                'role':    'assistant',
                'content': msg.content or '',
                'tool_calls': [
                    {
                        'id':   tc.id,
                        'type': 'function',
                        'function': {
                            'name':      tc.function.name,
                            'arguments': tc.function.arguments,
                        },
                    }
                    for tc in msg.tool_calls
                ],
            })

            for tc in msg.tool_calls:

                try:
                    tool_args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    tool_args = {}

                tool_result = dispatch_tool(tc.function.name, tool_args, self.coco_path)

                if verbose:
                    print(f'[Tool Result] {tool_result[:300]}...')

                messages.append({
                    'role':         'tool',
                    'tool_call_id': tc.id,
                    'content':      tool_result,
                })

            response = self.groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                tools=TOOL_DEF,
                tool_choice='auto',
                max_tokens=2048,
                temperature=0.1,
            )

        final_answer = response.choices[0].message.content or '(no response)'
        if verbose:
            print(f'\n[Answer]\n{final_answer}')
        return final_answer


print('RAG defined')


RAG defined


In [21]:
from pathlib import Path

if not Path(COCO_PATH).exists():
    raise FileNotFoundError(
        f'COCO file not found at {COCO_PATH}\n'
        'Make sure Phase 1 notebook has finished and saved to Drive.'
    )

auditor = RAGDataAuditor(coco_path=COCO_PATH, rebuild_index=True)
print('Auditor ready.')


[RAG] Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[RAG] Building index from COCO file...
[RAG] Built 810 chunks


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

[RAG] FAISS index: 810 vectors, dim=384
[RAG] Saved → /content/drive/MyDrive/augle_ai/data/rag/faiss.index
Auditor ready.


## Demo Queries


In [22]:
#  Demo Query 1
answer1 = auditor.query(
    'Are there any suspiciously small bounding boxes in the dataset? '
    'Flag anything under 500 square pixels and tell me which categories are affected.',
    verbose=True
)


Query: Are there any suspiciously small bounding boxes in the dataset? Flag anything under 500 square pixels and tell me which categories are affected.


[Retrieved 5 chunks]
  1. (score=0.406) Dataset dets: 832 images, 2897 annotations, 4 categories. Category distribution: person:1769, car:70...
  2. (score=0.320) Image dog_running_field_0014.jpg (id=184, 640x640px) contains 1 annotation(s): dog at (256,264) size...
  3. (score=0.307) Image dog_running_field_0015.jpg (id=185, 640x640px) contains 1 annotation(s): dog at (55,311) size ...
  4. (score=0.297) Image dog_running_field_0007.jpg (id=177, 640x640px) contains 1 annotation(s): dog at (260,233) size...
  5. (score=0.296) Image dog_running_field_0011.jpg (id=181, 640x640px) contains 1 annotation(s): dog at (245,47) size ...

[Tool Call] compute_bbox_stats({"category_filter":"","size_threshold_px":500})
[Tool Result] {
  "threshold_px": 500,
  "category_filter": "all",
  "per_category": {
    "person": {
      "count": 1769,
     

In [23]:
# Demo Query 2:
answer2 = auditor.query(
    'Is the dataset class-balanced? Show me the distribution of annotations per category '
    'and give a recommendation on whether I need to collect more data for any class.',
    verbose=True
)


Query: Is the dataset class-balanced? Show me the distribution of annotations per category and give a recommendation on whether I need to collect more data for any class.


[Retrieved 5 chunks]
  1. (score=0.515) Dataset dets: 832 images, 2897 annotations, 4 categories. Category distribution: person:1769, car:70...
  2. (score=0.354) Image 0002eed6350f1840.jpg (id=337, 640x640px) contains 1 annotation(s): person at (0,0) size 431x27...
  3. (score=0.352) Image 00056ff9ceb0c343.jpg (id=407, 640x640px) contains 1 annotation(s): person at (216,349) size 16...
  4. (score=0.352) Image 0004fca29aa55afd.jpg (id=397, 640x640px) contains 1 annotation(s): person at (90,0) size 549x6...
  5. (score=0.351) Image 000265efe8492bd9.jpg (id=316, 640x640px) contains 1 annotation(s): person at (11,0) size 212x2...

[Tool Call] compute_class_distribution(null)
[Tool Result] {
  "total_annotations": 2897,
  "num_categories": 4,
  "per_category": {
    "person": 1769,
    "car": 705,
    "dog": 416,
    

In [24]:
# Demo Query 3: RAG-only — general health report
answer3 = auditor.query(
    'Give me a high-level health report of this dataset: how many images, '
    'annotations, and categories are there, and what does the annotation density look like?',
    verbose=True
)


Query: Give me a high-level health report of this dataset: how many images, annotations, and categories are there, and what does the annotation density look like?


[Retrieved 5 chunks]
  1. (score=0.544) Dataset dets: 832 images, 2897 annotations, 4 categories. Category distribution: person:1769, car:70...
  2. (score=0.485) Image 00074503ceae5131.jpg (id=640, 640x640px) contains 1 annotation(s): dog at (0,175) size 639x464...
  3. (score=0.467) Image 001ecac90c6f4457.jpg (id=678, 640x640px) contains 1 annotation(s): dog at (193,241) size 259x1...
  4. (score=0.457) Image 0013ae186a0c98c8.jpg (id=656, 640x640px) contains 1 annotation(s): dog at (135,273) size 387x2...
  5. (score=0.455) Image 0002f32727ed756e.jpg (id=338, 640x640px) contains 1 annotation(s): person at (281,167) size 21...

[Tool Call] compute_class_distribution(null)
[Tool Result] {
  "total_annotations": 2897,
  "num_categories": 4,
  "per_category": {
    "person": 1769,
    "car": 705,
    "dog": 416,
    "person 